# PDF Extraction Agent - Google ADK

Notebook for the `PdfExtractionAgent` that extracts structured
legislative metadata from US legislative PDF documents using Google ADK + Gemini.

**Requirements:** `GOOGLE_API_KEY` (or `GEMINI_API_KEY`) environment variable must be set.

In [ ]:
import glob
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Load .env from project root and add it to sys.path
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "pdf_extraction" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
sys.path.insert(0, str(PROJECT_ROOT))

api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
assert api_key, "GOOGLE_API_KEY or GEMINI_API_KEY not set. Add it to your .env file."

PDF_DIR = (Path.cwd() / "pdfs") if Path.cwd().name == "pdf_extraction" else (Path.cwd() / "implementations" / "pdf_extraction" / "pdfs")
pdf_files = sorted(Path(p).name for p in glob.glob(str(PDF_DIR / "*.pdf")))
print(f"Found {len(pdf_files)} PDF(s): {pdf_files}")

## 1. Test the `read_pdf` tool standalone

In [ ]:
from aieng.agent_evals.pdf_extraction.tools import read_pdf

PDF_PATH = str(PDF_DIR / pdf_files[0])

result = read_pdf(PDF_PATH)
print(f"Status: {result['status']}")
print(f"Pages: {result['num_pages']}")
print(f"Content preview:\n{result['content'][:500]}")

## 2. Run the agent manually

In [ ]:
from aieng.agent_evals.pdf_extraction import PdfExtractionAgent

agent = PdfExtractionAgent()

In [ ]:
response = await agent.answer_async(
    pdf_path=PDF_PATH,
    jurisdiction="Idaho",
    prompt="Extract legislative metadata from this bill.",
)

print(f"Duration: {response.total_duration_ms}ms")
print(f"Tool calls: {response.tool_calls}")

## 3. Inspect output

In [ ]:
print(response.text)

In [ ]:
print("Reasoning chain:")
for i, step in enumerate(response.reasoning_chain, 1):
    print(f"  {i}. {step}")

print(f"\nToken usage: {agent.token_tracker.usage}")   

---
## 4. Interactive Gradio App

Run the cell below to launch an interactive UI for selecting PDFs and extracting metadata.

In [ ]:
# from implementations.pdf_extraction.demo import start_gradio_app

# start_gradio_app(enable_public_link=True)